In [6]:
#!pip install backtesting --quiet
#!pip install yfinance --quiet
#!pip install pandas_ta --quiet

In [2]:
try:
  import pandas as pd
  import numpy as np
  import yfinance as yf
  import pandas_ta as ta
  from backtesting import Backtest, Strategy
  from backtesting.lib import crossover
  from backtesting.test import SMA
  print("Libraries successfully imported!")
except:
  print("Error importing libraries")

Libraries successfully imported!


/usr/local/lib/python3.12/dist-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


In [3]:
df = yf.Ticker('BTC').history(start='2022-01-01',end='2026-01-01')

/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


In [4]:
def BB_lower(close, length, std):
    return ta.bbands(pd.Series(close), length=length, std=std).iloc[:, 0]

def BB_upper(close, length, std):
    return ta.bbands(pd.Series(close), length=length, std=std).iloc[:, 2]

def ATR(high, low, close, length):
    return ta.atr(pd.Series(high), pd.Series(low), pd.Series(close), length=length)

class BollingerBandStrategy(Strategy):
    sma_fast    = 10
    sma_slow    = 50
    bb_len      = 20
    bb_std      = 2
    atr_len     = 14
    atr_mult    = 2     # STOP_LOSS at 2xATR

    def init(self):
        c = self.data.Close

        self.sma_f = self.I(SMA, c, self.sma_fast)
        self.sma_s = self.I(SMA, c, self.sma_slow)

        self.bb_lo = self.I(BB_lower, c, self.bb_len, self.bb_std)
        self.bb_hi = self.I(BB_upper, c, self.bb_len, self.bb_std)

        self.atr = self.I(ATR, self.data.High, self.data.Low, self.data.Close, self.atr_len)


    def next(self):
        price = self.data.Close[-1]

        # Buy conditions
        trend_up = self.sma_f[-1] > self.sma_s[-1]
        hit_lower = price <= self.bb_lo[-1] * 1.03

        # Exit conditions
        trend_down = self.sma_f[-1] < self.sma_s[-1]
        hit_upper = price >= self.bb_hi[-1] * 1.03

        if not self.position:
            if trend_up or hit_lower:
              sl_price = price - self.atr_mult * self.atr[-1]   # Adaptive STOP_LOSS
              self.buy(sl=sl_price)


        else:
            if trend_down or hit_upper:
                self.position.close()

In [5]:
bt = Backtest(df, BollingerBandStrategy, cash=10_000, commission=.002, finalize_trades=True)
stats = bt.run()
print(stats)
#bt.plot()

Backtest.run:   0%|          | 0/307 [00:00<?, ?bar/s]

Start                     2024-07-31 00:00...
End                       2025-12-31 00:00...
Duration                    518 days 01:00:00
Exposure Time [%]                    62.18487
Equity Final [$]                  17719.62454
Equity Peak [$]                    21457.7696
Commissions [$]                    1702.85423
Return [%]                           77.19625
Buy & Hold Return [%]                43.44444
Return (Ann.) [%]                    49.75442
Volatility (Ann.) [%]                52.04739
CAGR [%]                             32.08699
Sharpe Ratio                          0.95594
Sortino Ratio                         2.51258
Calmar Ratio                          2.43364
Alpha [%]                            56.98852
Beta                                  0.46514
Max. Drawdown [%]                   -20.44446
Avg. Drawdown [%]                    -6.38733
Max. Drawdown Duration      141 days 00:00:00
Avg. Drawdown Duration       28 days 00:00:00
# Trades                          